# Support Vector Regression: Fuel Economy with an Error Tube

**Family:** Supervised learning | **Level:** Intermediate | **Dataset:** `auto_mpg` | **Runs on:** CPU, about 1 minute(s)

Fit a flexible kernel regressor that ignores small errors.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- What the epsilon insensitive loss is
- How the RBF kernel gives nonlinear fits
- How C, epsilon and gamma interact

**Prerequisites:**

- Linear regression notebook
- Train, validation and test splits

**Skills you will have afterwards:**

- Choose a hyperparameter on validation data
- Compare with a baseline
- Run a sensitivity experiment

## 2. Real-World Problem

**Context.** Fuel economy depends on weight, power and year in a nonlinear way.

**The problem.** Predict mpg with a model that can bend, without hand crafting polynomial terms.

**Why it matters.** Better predictions improve design studies and fleet planning.

**Why machine learning.** The shape of the relation is unknown.

**Why this algorithm.** SVR is a strong kernel method for small and medium tabular data and ignores errors inside a tolerance tube.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### A tutorial on support vector regression

- **Type:** foundational paper
- **Authors:** Smola, A. J. and Scholkopf, B.
- **Year and venue:** 2004, Statistics and Computing 14(3):199-222
- **DOI or URL:** https://doi.org/10.1023/B:STCO.0000035301.49549.88
- **Verification:** Crossref 2026-11-17
- **Research objective:** Review support vector regression.
- **Methodology:** Epsilon insensitive loss with kernels and a flat function preference.
- **Dataset:** Tutorial, small illustrations only.
- **Main findings:** Sparse kernel regression with a controllable error tolerance.
- **Limitations:** Cost grows quickly with rows. Needs scaling and parameter tuning.

### Support-vector networks

- **Type:** foundational paper
- **Authors:** Cortes, C. and Vapnik, V.
- **Year and venue:** 1995, Machine Learning 20(3):273-297
- **DOI or URL:** https://doi.org/10.1007/BF00994018
- **Verification:** Crossref 2026-11-17
- **Research objective:** Introduce the soft margin support vector machine for non separable data.
- **Methodology:** Maximise the margin in a kernel feature space with slack variables.
- **Dataset:** Handwritten digit recognition (USPS and NIST data) and synthetic examples.
- **Main findings:** Polynomial kernel machines matched the best published digit classifiers of the time.
- **Limitations:** Training cost grows quickly with the number of rows. Needs feature scaling and kernel tuning.

**Relationship to this notebook.** Smola and Scholkopf review support vector regression. Cortes and Vapnik introduced the classification machine from which it descends. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Epsilon insensitive loss.** $L_\varepsilon(y, f) = \max(0, |y - f| - \varepsilon)$.

**Primal problem.**

$$\min_{w,b,\xi,\xi^*}\; \tfrac12 \lVert w \rVert^2 + C\sum_i (\xi_i + \xi_i^*) \quad \text{s.t.}\quad |y_i - w^\top\phi(x_i) - b| \le \varepsilon + \xi_i$$

**Prediction.** $f(x) = \sum_i (\alpha_i - \alpha_i^*) K(x_i, x) + b$ with the RBF kernel $K(x,x') = \exp(-\gamma \lVert x - x' \rVert^2)$.

**Assumptions.** Smooth target, features on comparable scales.

**Limitations.** Training time grows between quadratically and cubically with rows. Needs scaling and tuning of $C$, $\varepsilon$, $\gamma$.

## 5. Dataset Introduction

**Auto MPG** (Quinlan, 1993), UCI Machine Learning Repository, licence CC BY 4.0, DOI 10.24432/C5859H. 392 cars from 1970 to 1982 without missing horsepower (as shipped in `mlxtend`). Target `mpg`. Features: `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year`, `origin` (1 US, 2 Europe, 3 Japan). All numeric. Limit: old cars only, so conclusions do not transfer to modern vehicles.

The table printed below the next cell comes from the dataset registry in `src/ml_lab/data/registry.py`. Licence terms belong to the data provider, not to this repository.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from ml_lab.data import load_dataset, missing_report
from ml_lab.preprocessing import split_three_way, build_preprocessor, make_model_pipeline, feature_names
from ml_lab.evaluation import evaluate_estimator, compare_models
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "support_vector_regression"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

In [ ]:
bundle = load_dataset("auto_mpg")
data = bundle.frame
TASK = bundle.task
TARGET = bundle.target
print("task:", TASK, "| target:", TARGET, "| shape:", data.shape)
display(pd.Series(bundle.describe_source(), name="source").to_frame())
display(data.head())

## 6. Exploratory Data Analysis

We look at size, types, missing values, the target, feature distributions, correlation and outliers before any modelling. 

In [ ]:
print("Rows and columns:", data.shape)
display(data.dtypes.value_counts().rename("columns by dtype").to_frame())
display(data.describe().T.round(3).head(20))
missing = missing_report(data)
print("Columns with missing values:", len(missing))
display(missing)
print("Duplicate rows:", int(data.duplicated().sum()))

In [ ]:
fig = viz.plot_target(data[TARGET], TASK)
plt.show()
fig = viz.plot_distributions(data.drop(columns=[TARGET]))
plt.show()
fig = viz.plot_correlation(data)
plt.show()
display(viz.outlier_table(data.drop(columns=[TARGET])).head(8))

## 7. Data Preprocessing

Rules used here:

- Split first, then fit imputers, scalers and encoders on the training rows only. This prevents data leakage.
- The test set is touched once, at the end.
- Order inside the pipeline: impute, scale or encode, then model.


In [ ]:

X, y = data.drop(columns=[TARGET]), data[TARGET]
X_train, X_val, X_test, y_train, y_val, y_test = split_three_way(X, y, task=TASK, seed=SEED)
print("train / validation / test rows:", len(X_train), len(X_val), len(X_test))
if TASK == "classification":
    split_rates = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                                "validation": y_val.value_counts(normalize=True),
                                "test": y_test.value_counts(normalize=True)}).round(3)
    display(split_rates)
prep = build_preprocessor(X_train, scale=True, encode="onehot")
Xtr = prep.fit_transform(X_train)   # fit on train only
Xva, Xte = prep.transform(X_val), prep.transform(X_test)
print("transformed train shape:", Xtr.shape)

## 8. Algorithm Implementation

### 8.1 Concept

Only points on or outside the tube become support vectors, so the model is sparse in the data. Larger $C$ punishes errors more and gives a wigglier fit.

### 8.2 Python implementation from scratch

Not implemented from scratch: solving the dual quadratic program is a numerical optimisation task that adds little beyond understanding the loss. We plot the loss instead.

In [ ]:
r = np.linspace(-3, 3, 200)
plt.plot(r, r ** 2, label="squared loss")
plt.plot(r, np.abs(r), label="absolute loss")
plt.plot(r, np.maximum(0, np.abs(r) - 0.5), label="epsilon insensitive (epsilon=0.5)")
plt.xlabel("Residual (mpg)"); plt.ylabel("Loss"); plt.legend(); plt.title("Loss functions"); plt.show()

### 8.3 Library implementation

The estimator is wrapped in a pipeline so imputation, scaling and encoding are fitted on training rows only.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `C` | Cost of errors outside the tube. |
| `epsilon` | Tube half width in target units. |
| `gamma` | RBF width. `scale` uses 1 / (n_features * variance). |
| `kernel` | `rbf`, `linear`, `poly`. |

## 9. Model Training

Procedure: fit on the training rows, choose one hyperparameter on the validation rows, refit on training plus validation, then evaluate once on the test rows. The random seed is fixed (`SEED = 42`). 

In [ ]:
SELECT = "RMSE"
LOWER_IS_BETTER = True
SCORING = "neg_root_mean_squared_error"
SWEEP_PARAM = "model__C"
SWEEP_VALUES = [0.1, 1, 3, 10, 30, 100, 300]

def build_pipeline():
    return make_model_pipeline(X_train, SVR(kernel="rbf", epsilon=0.5, gamma="scale"), scale=True, encode="onehot")

pipe = build_pipeline()
print(pipe.named_steps["model"])
rows = []
for value in SWEEP_VALUES:
    candidate = clone(pipe).set_params(**{SWEEP_PARAM: value}).fit(X_train, y_train)
    rows.append({SWEEP_PARAM: value, **evaluate_estimator(candidate, X_val, y_val, TASK)})
sweep = pd.DataFrame(rows).set_index(SWEEP_PARAM)
display(sweep[[c for c in [SELECT, "R2", "MAE", "Accuracy", "F1", "ROC_AUC", "PR_AUC"] if c in sweep.columns]].round(4))
best_value = sweep[SELECT].idxmin() if LOWER_IS_BETTER else sweep[SELECT].idxmax()
print("best", SWEEP_PARAM, "on validation:", best_value)
final_model = clone(pipe).set_params(**{SWEEP_PARAM: best_value})
X_trainval, y_trainval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)   # refit before test
print("refit on train plus validation rows")

## 10. Evaluation

Regression metrics: MAE (average absolute error in target units), RMSE (square root of the mean squared error, same unit, punishes large errors), R-squared (share of variance explained, can be negative on new data) and MAPE when the target is never zero.

A dummy model gives the floor every real model must beat.

In [ ]:
baseline = DummyRegressor(strategy="mean") if TASK == "regression" else DummyClassifier(strategy="prior")
baseline = make_model_pipeline(X_train, baseline, scale=False).fit(X_trainval, y_trainval)
test_metrics = evaluate_estimator(final_model, X_test, y_test, TASK)
baseline_metrics = evaluate_estimator(baseline, X_test, y_test, TASK)
print("Test set, scored once after model choice")
display(compare_models({"baseline": baseline_metrics, "SVR": test_metrics}))
cv = cross_validate(final_model, X_trainval, y_trainval, cv=5, scoring=SCORING, n_jobs=1)
cv_scores = cv["test_score"]
print(f"5-fold CV {SCORING}: mean {cv_scores.mean():.4f}, std {cv_scores.std(ddof=1):.4f}")
record = ExperimentRecord(notebook=NOTEBOOK_ID, family="supervised", algorithm="SVR", dataset="auto_mpg",
                          task=TASK, seed=SEED, config={"param": SWEEP_PARAM, "value": str(best_value)},
                          metrics={k: float(v) for k, v in test_metrics.items()},
                          extra={"baseline": {k: float(v) for k, v in baseline_metrics.items()},
                                 "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std(ddof=1)),
                                 "cv_scoring": SCORING, "select_metric": SELECT})
print("saved", record.save().name)

## 11. Visualization

All plots use the test rows unless the title says otherwise.

In [ ]:
pred = final_model.predict(X_test)
fig = viz.plot_pred_vs_actual(y_test, pred, unit="(miles per gallon)")
save_figure(fig, NOTEBOOK_ID, "pred_vs_actual"); plt.show()
fig = viz.plot_residuals(y_test, pred, unit="(miles per gallon)")
save_figure(fig, NOTEBOOK_ID, "residuals"); plt.show()
fig = viz.plot_learning_curve(final_model, X_trainval, y_trainval, scoring=SCORING)
save_figure(fig, NOTEBOOK_ID, "learning_curve"); plt.show()

## 12. Interpretation

Permutation importance shuffles one feature and measures how much the error grows. Correlated features share importance, so read it as a ranking of groups. Check the residual plot for curvature or funnel shapes, which mean the model misses structure or the error variance changes with the prediction.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring=SCORING, n_repeats=10, random_state=SEED)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
fig = viz.plot_importance(importance.index, importance.values, title="Permutation importance on the test set",
                          top=12, xlabel="Drop in " + SCORING + " when shuffled")
save_figure(fig, NOTEBOOK_ID, "importance"); plt.show()
print("Top features:", ", ".join(importance.index[:5]))
errors = (y_test.values - final_model.predict(X_test)) if TASK == "regression" else (y_test.values != final_model.predict(X_test))
if TASK == "regression":
    worst = pd.DataFrame({"actual": y_test.values, "predicted": final_model.predict(X_test)}, index=y_test.index)
    worst["abs_error"] = (worst["actual"] - worst["predicted"]).abs()
    print("Largest test errors:")
    display(worst.sort_values("abs_error", ascending=False).head(5).round(3))
else:
    print("Misclassified test rows:", int(errors.sum()), "of", len(errors))
print()
print(f"Test {SELECT}: {test_metrics[SELECT]:.4f} | baseline {SELECT}: {baseline_metrics[SELECT]:.4f}")
print(f"Cross validation spread (std): {cv_scores.std(ddof=1):.4f}. A test gap larger than this spread is meaningful.")

## 13. Experiments and Challenges

### Experiment 1: Epsilon and support vectors

Larger epsilon means fewer support vectors and a smoother fit.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_rmse(model, X_data=None, y_data=None):
    scores = cross_val_score(model, X_data if X_data is not None else X_trainval,
                             y_data if y_data is not None else y_trainval,
                             cv=5, scoring="neg_root_mean_squared_error")
    return -scores.mean(), scores.std(ddof=1)
for eps in [0.1, 0.5, 1.0, 2.0, 4.0]:
    m = clone(final_model).set_params(model__epsilon=eps).fit(X_trainval, y_trainval)
    mean, std = cv_rmse(clone(final_model).set_params(model__epsilon=eps))
    print(f"epsilon={eps}: support vectors {m.named_steps['model'].support_.size} of {len(X_trainval)}, CV RMSE {mean:.2f} +- {std:.2f}")

### Experiment 2: Kernel choice

Compare linear, polynomial and RBF kernels with cross validation.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_rmse(model, X_data=None, y_data=None):
    scores = cross_val_score(model, X_data if X_data is not None else X_trainval,
                             y_data if y_data is not None else y_trainval,
                             cv=5, scoring="neg_root_mean_squared_error")
    return -scores.mean(), scores.std(ddof=1)
for kernel in ["linear", "poly", "rbf"]:
    mean, std = cv_rmse(clone(final_model).set_params(model__kernel=kernel))
    print(f"{kernel}: CV RMSE {mean:.2f} +- {std:.2f}")

### Experiment 3: Gamma sensitivity

A very large gamma memorises points. Compare train and validation error.

In [ ]:
rows = []
for gamma in [0.001, 0.01, 0.1, 1, 10]:
    m = clone(final_model).set_params(model__gamma=gamma).fit(X_train, y_train)
    rows.append({"gamma": gamma, "train_RMSE": evaluate_estimator(m, X_train, y_train, TASK)["RMSE"], "val_RMSE": evaluate_estimator(m, X_val, y_val, TASK)["RMSE"]})
display(pd.DataFrame(rows).set_index("gamma").round(3))

### Exercises for learners

1. Repeat the notebook with `SEED = 7` and report how much the headline metric changes.
2. Add one more feature or remove one and explain the change using cross validation.
3. Replace the validation sweep with `GridSearchCV` and compare the chosen value.

## 14. Research Discussion

Smola and Scholkopf describe the epsilon tube and sparsity. Our epsilon experiment shows the support vector count shrinking as epsilon grows, matching the described mechanism on different data.

**Dataset and method differences.** The papers above use other datasets, other splits and often other implementations. We do not claim that our numbers agree or disagree with them. What this notebook can show is whether the behaviour described in the paper (for example the effect of a parameter) appears on our data.

**Possible improvements.** Tune C, epsilon and gamma jointly with random search, and compare with gradient boosting on the same folds.

**Research limitations.** One dataset, one split seed for the headline number, and small test sets for the UCI data. Cross validation spread is reported to show how unstable a single split can be.

## 15. Summary and Further Reading

**Key takeaways**

- Scale features before SVR.
- Epsilon controls sparsity.
- Gamma and C control flexibility.

**Use cases.** Small and medium nonlinear regression problems.

**Limitations.** Poor scaling to large data, no native uncertainty.

**Recommended next notebook.** Gradient Boosting Regressor.

**Further reading.** See `references.md` in this folder and the library documentation for the classes used above.